# BugsInPy — Plausibility Testing

Runs the actual project tests (via the BugsInPy framework) against generated patches and reports **plausible@K**.

**Inputs:**
- `data/bugsinpy_eval.jsonl` — eval set (196 single-function bugs)
- `results/bugsinpy_<model>_baseline.jsonl` — generations from inference

**Output:** `results/bugsinpy_<model>_plausibility.jsonl` — one line per (bug_id, gen_idx) with test_pass / test_status.

**Parameters:** `START_BUG`, `END_BUG` — slice the eval set by index. **Use these to chunk the work across multiple Colab sessions** (e.g. session 1: 0..50, session 2: 50..100, ...). Resume is automatic — rerunning skips already-tested (bug, gen) pairs.

**Time:** Highly variable. Per bug: 30s-25min depending on project (pandas/scipy/keras = slow, thefuck/youtube-dl = fast). Plan **~1-2 hours per 25 bugs** as a rough estimate.

**Runtime:** Use **Colab CPU (no GPU needed)**. GPU runtime is wasted here — tests are I/O and CPU bound.

## 1. Clone repo & set paths

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"
REPO_DIR = "/content/snake-repairllama-baseline"
BUGSINPY_DIR = "/content/BugsInPy"
PYENV_ROOT   = "/content/.pyenv"
WORK_ROOT    = "/content/bugsinpy_work"     # per-bug checkouts live here

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())

## 2. One-time environment setup

Installs apt build deps, clones BugsInPy, installs **pyenv** and Python 3.6 / 3.7 / 3.8 (the versions most BugsInPy projects need).

**~15-25 min on first run.** Idempotent — re-running is cheap.

In [ ]:
!bash scripts/setup_bugsinpy.sh {BUGSINPY_DIR} {PYENV_ROOT}

## 3. Get generations file (mount Drive or upload)

In [ ]:
from google.colab import drive
import shutil, os
drive.mount("/content/drive", force_remount=False)

DRIVE_GENS = "/content/drive/MyDrive/snake-repairllama-baseline-results/bugsinpy_codellama_baseline.jsonl"
LOCAL_GENS = "results/bugsinpy_codellama_baseline.jsonl"

if os.path.exists(DRIVE_GENS):
    os.makedirs("results", exist_ok=True)
    shutil.copy(DRIVE_GENS, LOCAL_GENS)
    print("copied:", LOCAL_GENS)
else:
    print("WARN: not found on Drive — upload manually or change the path.")
    os.makedirs("results", exist_ok=True)

## 4. Parameters

Edit, then run section 5. The eval set is 196 bugs (indices 0..195) — chunk the work however you like.

**Recommended cadence:** 25-50 bugs per session, save back to Drive between runs.

In [ ]:
EVAL_FILE        = "data/bugsinpy_eval.jsonl"
GENERATIONS_FILE = "results/bugsinpy_codellama_baseline.jsonl"  # change me if your model name differs
OUTPUT_FILE      = "results/bugsinpy_codellama_plausibility.jsonl"

START_BUG = 0     # i  (inclusive)
END_BUG   = 25    # j  (exclusive). Total set is 196.

TIMEOUT_COMPILE = 1200   # 20 min — large projects need this
TIMEOUT_TEST    =  180   #  3 min per test run
SKIP_COMPILE_FAILURES = True   # if compile fails, mark all 10 gens as compile_failed and move on
RESUME = True

# Sanity
assert os.path.exists(EVAL_FILE), f"missing: {EVAL_FILE}"
assert os.path.exists(GENERATIONS_FILE), f"missing: {GENERATIONS_FILE}"
print("Inputs OK. Will test bugs", START_BUG, "..", END_BUG)

## 5. Run plausibility on the slice

**Watch the streaming output:** each bug logs `checkout`, `compile`, then per-patch `✓` (pass) / `✗` (fail/error/timeout). Cell streams progress so you can monitor a Colab session that may disconnect.

In [ ]:
from src.runners.bugsinpy import run_plausibility

run_plausibility(
    eval_jsonl=EVAL_FILE,
    inference_jsonl=GENERATIONS_FILE,
    output_jsonl=OUTPUT_FILE,
    bugsinpy_dir=BUGSINPY_DIR,
    work_root=WORK_ROOT,
    pyenv_root=PYENV_ROOT,
    start_bug=START_BUG,
    end_bug=END_BUG,
    timeout_sec_compile=TIMEOUT_COMPILE,
    timeout_sec_test=TIMEOUT_TEST,
    resume=RESUME,
    skip_compile_failures=SKIP_COMPILE_FAILURES,
)

## 6. Save partial results to Drive (do this between runs!)

In [ ]:
import shutil, os
dest = "/content/drive/MyDrive/snake-repairllama-baseline-results"
os.makedirs(dest, exist_ok=True)
shutil.copy(OUTPUT_FILE, dest + "/" + os.path.basename(OUTPUT_FILE))
print("saved to Drive:", dest + "/" + os.path.basename(OUTPUT_FILE))

## 7. Aggregate report (over everything tested so far)

In [ ]:
from src.metrics import evaluate_file, print_report

result = evaluate_file(
    inference_jsonl=GENERATIONS_FILE,
    eval_jsonl=EVAL_FILE,
    plausibility_jsonl=OUTPUT_FILE,
)
print_report("BugsInPy — vanilla CodeLlama-7B-Python (with plausibility)", result)

## 8. (Optional) Per-bug breakdown

Useful for spotting projects/bugs where compile or test consistently fails — those are usually environment issues, not real model failures.

In [ ]:
import json
from collections import Counter

by_bug = {}
with open(OUTPUT_FILE) as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        by_bug.setdefault(r["bug_id"], []).append(r)

print(f"{'bug_id':40s}  {'pass/total':>10s}  status_counts")
print("-" * 100)
for bug_id in sorted(by_bug):
    rows = by_bug[bug_id]
    n_pass = sum(r['test_pass'] for r in rows)
    statuses = Counter(r['test_status'] for r in rows)
    status_str = " ".join(f"{s}={c}" for s, c in statuses.most_common())
    print(f"{bug_id:40s}  {n_pass:>4}/{len(rows):>4}    {status_str}")

## Tips & known gotchas

- **Disk space:** each project checkout is 50 MB - 1 GB. Colab gives ~80 GB; running ~50-80 different bugs at once is fine but watch `df -h /content`.
- **Compile failures are common** for older Python (2.7) or projects with native deps (scipy, numpy<1.18). They're marked `compile_failed` in the output and don't block the rest.
- **If a session disconnects mid-bug**, the partially-tested bug's gens may already be in the output file. The resume logic skips them. Wasted work = at most 10 patches × 1 bug = a few minutes.
- **To re-test a bug from scratch** (e.g. after fixing the runner), delete its rows from the output JSONL or pick a new OUTPUT_FILE name.
- **Don't run on a GPU runtime** — wasteful, no benefit. Use the **CPU** runtime.